# Stage 2: load stored results and redraw the paper figures

This notebook does not rerun GPU jobs. It loads the stored per-replication CSVs, rebuilds the summaries in Python, checks that they match the cached summaries, and writes the paper filenames into `figures/`.

| Paper label | Output file |
|---|---|
| `fig:teaser` | `picpi_teaser_green.pdf` |
| `fig:thm52_box_rate` | `thm52_width_mean_boxplot.pdf`, `thm52_width_vs_rate.pdf` |
| `fig:emp_mode_diagnostics` | `heldout_picpi_failure_rate_by_method.pdf`, `heldout_picpi_pass_rate_by_interval_length.pdf` |
| `fig:task1_interval_visualization` | `task1_interval_visualization.pdf` |
| `fig:task1_misspecified_tree` | `task1_misspecified_point_estimator_decision_tree.pdf` |
| `tab:task1_multivariate_mc_summary` | `task1_multivariate_mc_summary.tex` |
| `fig:task2` | `task2_multiclass_dgp_sweep.pdf` |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from picpi.aggregate import (
    load_and_verify_empirical,
    load_and_verify_task1,
    load_and_verify_task2,
    load_and_verify_thm52,
)
from picpi.paths import figures_dir
from picpi.plot import (
    plot_empirical_mode,
    plot_task1_tree,
    plot_task1_visualization,
    plot_task2,
    plot_teaser,
    plot_thm52,
    write_task1_table,
)
from picpi.teaser import load_teaser

OUT = figures_dir()
OUT

## Figure teaser (`fig:teaser`)

Original compute: `experiments/uncertainty_quantification/teaser.ipynb`.
This folder: `picpi/teaser.py` stores the points and intervals; `plot_teaser` draws the figure.

In [ ]:
teaser = load_teaser()
plot_teaser(teaser, OUT)

## Figure width shrinkage (`fig:thm52_box_rate`)

Uses the saved GPU intermediates in `cached_results/thm52/thm52_df_results.csv` (900 rows). The summary is rebuilt here and checked against `thm52_summary.csv`.

In [ ]:
thm52_df, thm52_summary = load_and_verify_thm52()
print("rebuilt thm52 summary matches cached summary")
plot_thm52(thm52_df, thm52_summary, OUT)

## Figure empirical-mode diagnostics (`fig:emp_mode_diagnostics`)

Uses the saved GPU intermediates `heldout_picpi_results.csv` and `heldout_picpi_interval_results.csv`. Summaries are rebuilt and checked against the cached summary CSVs.

In [ ]:
emp_summary, emp_width = load_and_verify_empirical()
print("rebuilt empirical summaries match cached summaries")
plot_empirical_mode(emp_summary, emp_width, OUT)

## Task 1 figures and table

Original compute: `experiments/final_sub/task1_final_submission.ipynb`.
The Monte Carlo table is rebuilt from `multivariate_mc_reps.csv`.

In [ ]:
task1 = load_and_verify_task1()
print(plot_task1_visualization(task1["univariate"], OUT))
print(plot_task1_tree(task1["tree"], OUT))
print(write_task1_table(task1["mc_summary"], OUT))
task1["mc_summary"]

## Figure Task 2 (`fig:task2`)

Uses the saved CPU sweep intermediates `cached_results/task2/*/long.csv` (1,000 seeds). Summaries are rebuilt and checked against `summary.csv`.

In [ ]:
task2 = load_and_verify_task2()
print("rebuilt Task 2 summaries match cached summaries")
plot_task2(task2, OUT)